# StrokeGuard: Predição de Risco de AVC com Machine Learning

## Contexto do Problema

O Acidente Vascular Cerebral (AVC) é uma das principais causas de morte e incapacidade no mundo. Segundo a Organização Mundial da Saúde, o AVC é responsável por aproximadamente 11% de todas as mortes globais. A identificação precoce de indivíduos com alto risco pode salvar vidas através de intervenções preventivas.

## Objetivo

Construir um modelo de classificação capaz de prever o risco de AVC em pacientes com base em dados clínicos e demográficos. Avaliaremos diferentes algoritmos de Machine Learning e compararemos suas performances.

## Dataset

Utilizamos o *Healthcare Dataset Stroke Data*, que contém **5.110 registros** com **12 colunas**:
- **Identificação:** id
- **Características demográficas:** gender, age, ever_married, work_type, Residence_type
- **Condições de saúde:** hypertension, heart_disease, avg_glucose_level, bmi, smoking_status
- **Variável alvo:** stroke (0 = sem AVC, 1 = com AVC)

O dataset apresenta desbalanceamento de classes significativo, com apenas ~5% de casos positivos (AVC).

## 2. Importação de Bibliotecas

Importamos todas as bibliotecas necessárias para manipulação de dados, visualização e modelagem.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    recall_score, f1_score, roc_auc_score
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("Todas as bibliotecas importadas com sucesso!")

## 3. Carregamento dos Dados

Carregamos o dataset a partir do repositório GitHub. Caso a URL não esteja acessível, tentamos o caminho local.

In [ ]:
url = "https://raw.githubusercontent.com/AldySouza/mvp-sprint-2-puc-rio/master/data/healthcare-dataset-stroke-data.csv"
local_path = "../data/healthcare-dataset-stroke-data.csv"

try:
  df = pd.read_csv(url)
  print(f"Dataset carregado da URL remota.")
except Exception:
  df = pd.read_csv(local_path)
  print(f"Dataset carregado do caminho local.")
print(f"Shape: {df.shape}")
print(f"\nTipos de dados:\n{df.dtypes}")

df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## 4. Análise Exploratória dos Dados (EDA)

### 4.1 Distribuição da Variável Alvo

Verificamos o desbalanceamento de classes na variável `stroke`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['stroke'].value_counts()
sns.countplot(data=df, x='stroke', ax=ax)
ax.set_title('Distribuição da Variável Alvo (stroke)')
ax.set_xlabel('Stroke')
ax.set_ylabel('Contagem')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

total = len(df)
print("Distribuição de classes:")
for val, cnt in counts.items():
    print(f"  stroke={val}: {cnt} ({cnt/total*100:.1f}%)")

### 4.2 Valores Ausentes

Identificamos e quantificamos os valores ausentes no dataset.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, '%': missing_pct})
print("Valores ausentes por coluna:")
print(missing_df[missing_df['Ausentes'] > 0])
print(f"\nTotal de registros com bmi ausente: {df['bmi'].isnull().sum()}")

### 4.3 Distribuição das Features Numéricas

In [ ]:
numerical_cols = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, hue='stroke', kde=True, ax=axes[i], bins=30)
    axes[i].set_title(f'Distribuição de {col}')
plt.tight_layout()
plt.show()

### 4.4 Distribuição das Features Categóricas

In [ ]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, hue='stroke', ax=axes[i])
    axes[i].set_title(f'Distribuição de {col}')
    axes[i].tick_params(axis='x', rotation=45)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

### 4.5 Mapa de Correlação

Analisamos a correlação entre as features numéricas e a variável alvo.

In [ ]:
corr_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'stroke']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Mapa de Correlação')
plt.tight_layout()
plt.show()

## 5. Preparação dos Dados

Removemos a coluna `id` (irrelevante), definimos os grupos de features e separamos em treino/teste com estratificação.

In [ ]:
df = df.drop(columns=['id'])

numerical_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
passthrough_features = ['hypertension', 'heart_disease']

X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de treino: {X_train.shape[0]} amostras")
print(f"Conjunto de teste:  {X_test.shape[0]} amostras")
print(f"\nDistribuição no treino:")
print(y_train.value_counts(normalize=True).round(4))
print(f"\nDistribuição no teste:")
print(y_test.value_counts(normalize=True).round(4))

## 6. Demonstração de Normalização vs Padronização

### Diferença conceitual

- **Normalização (MinMaxScaler):** Redimensiona os valores para o intervalo [0, 1]. Útil quando a distribuição não é necessariamente gaussiana e o algoritmo é sensível à escala (ex: KNN, redes neurais).

- **Padronização (StandardScaler):** Transforma os dados para média 0 e desvio padrão 1. Mais adequada quando os dados seguem distribuição aproximadamente normal e para algoritmos como SVM.

Abaixo demonstramos o efeito da normalização MinMaxScaler. No pipeline final utilizaremos StandardScaler.

In [ ]:
X_train_demo = X_train[numerical_features].copy()

scaler_minmax = MinMaxScaler()
X_train_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(X_train_demo),
    columns=numerical_features,
    index=X_train_demo.index
)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, col in enumerate(numerical_features):
    axes[0, i].hist(X_train_demo[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{col} — Original')
    axes[1, i].hist(X_train_normalized[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[1, i].set_title(f'{col} — MinMaxScaler [0, 1]')
plt.suptitle('Antes e Depois da Normalização (MinMaxScaler)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Estatísticas após MinMaxScaler:")
print(X_train_normalized.describe().round(4))

## 7. Pipeline de Pré-processamento

Construímos um `ColumnTransformer` que:
- **Numéricas:** Imputa valores ausentes (mediana) + padronização (StandardScaler)
- **Categóricas:** One-Hot Encoding
- **Passthrough:** hypertension e heart_disease são mantidas como estão (já binárias)

In [ ]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('pass', 'passthrough', passthrough_features)
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Shape original X_train:      {X_train.shape}")
print(f"Shape transformado X_train:   {X_train_transformed.shape}")
print(f"Shape transformado X_test:    {X_test_transformed.shape}")

## 8. Treinamento — K-Nearest Neighbors (KNN)

O KNN classifica uma amostra com base na classe majoritária dos K vizinhos mais próximos. Testamos diferentes combinações de hiperparâmetros via GridSearchCV.

In [ ]:
knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

knn_param_grid = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn_grid = GridSearchCV(
    knn_pipeline, knn_param_grid, cv=cv,
    scoring='f1', n_jobs=-1, verbose=0
)
knn_grid.fit(X_train, y_train)

print("Melhores hiperparâmetros (KNN):")
print(knn_grid.best_params_)
print(f"\nMelhor F1-score (CV): {knn_grid.best_score_:.4f}")

In [ ]:
y_pred_knn = knn_grid.predict(X_test)

print("Relatório de Classificação — KNN:")
print(classification_report(y_test, y_pred_knn, digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Sem AVC', 'Com AVC'],
            yticklabels=['Sem AVC', 'Com AVC'])
ax.set_title('Matriz de Confusão — KNN')
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## 9. Treinamento — Árvore de Decisão

A Árvore de Decisão cria regras de divisão hierárquicas baseadas nas features. É interpretável e não exige normalização, mas pode sofrer overfitting.

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

dt_param_grid = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    dt_pipeline, dt_param_grid, cv=cv,
    scoring='f1', n_jobs=-1, verbose=0
)
dt_grid.fit(X_train, y_train)

print("Melhores hiperparâmetros (Árvore de Decisão):")
print(dt_grid.best_params_)
print(f"\nMelhor F1-score (CV): {dt_grid.best_score_:.4f}")

In [ ]:
y_pred_dt = dt_grid.predict(X_test)

print("Relatório de Classificação — Árvore de Decisão:")
print(classification_report(y_test, y_pred_dt, digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Sem AVC', 'Com AVC'],
            yticklabels=['Sem AVC', 'Com AVC'])
ax.set_title('Matriz de Confusão — Árvore de Decisão')
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## 10. Treinamento — Naive Bayes (GaussianNB)

O Naive Bayes assume independência entre features e é eficiente computacionalmente. O GaussianNB assume que as features contínuas seguem distribuição normal.

In [ ]:
nb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

nb_param_grid = {
    'classifier__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

nb_grid = GridSearchCV(
    nb_pipeline, nb_param_grid, cv=cv,
    scoring='f1', n_jobs=-1, verbose=0
)
nb_grid.fit(X_train, y_train)

print("Melhores hiperparâmetros (Naive Bayes):")
print(nb_grid.best_params_)
print(f"\nMelhor F1-score (CV): {nb_grid.best_score_:.4f}")

In [ ]:
y_pred_nb = nb_grid.predict(X_test)

print("Relatório de Classificação — Naive Bayes:")
print(classification_report(y_test, y_pred_nb, digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_nb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Sem AVC', 'Com AVC'],
            yticklabels=['Sem AVC', 'Com AVC'])
ax.set_title('Matriz de Confusão — Naive Bayes')
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## 11. Treinamento — Support Vector Machine (SVM)

O SVM busca o hiperplano que melhor separa as classes, maximizando a margem. Funciona bem em espaços de alta dimensionalidade e com o kernel RBF pode capturar relações não-lineares.

In [ ]:
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, random_state=42, class_weight='balanced'))
])

svm_param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

svm_grid = GridSearchCV(
    svm_pipeline, svm_param_grid, cv=cv,
    scoring='f1', n_jobs=-1, verbose=0
)
svm_grid.fit(X_train, y_train)

print("Melhores hiperparâmetros (SVM):")
print(svm_grid.best_params_)
print(f"\nMelhor F1-score (CV): {svm_grid.best_score_:.4f}")

In [ ]:
y_pred_svm = svm_grid.predict(X_test)

print("Relatório de Classificação — SVM:")
print(classification_report(y_test, y_pred_svm, digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Sem AVC', 'Com AVC'],
            yticklabels=['Sem AVC', 'Com AVC'])
ax.set_title('Matriz de Confusão — SVM')
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## 12. Comparação dos Modelos

Coletamos as métricas de todos os modelos e identificamos o melhor considerando múltiplas métricas simultaneamente. Para que um modelo seja elegível como "melhor", ele precisa atender aos requisitos mínimos de desempenho definidos para implantação (acurácia, recall e F1-score). Entre os modelos que atendem a esses requisitos, selecionamos aquele com o maior F1-score para a classe positiva (AVC).

In [ ]:
models = {
    'KNN': (knn_grid, y_pred_knn),
    'Árvore de Decisão': (dt_grid, y_pred_dt),
    'Naive Bayes': (nb_grid, y_pred_nb),
    'SVM': (svm_grid, y_pred_svm)
}

results = []
for name, (grid, y_pred) in models.items():
    y_proba = grid.predict_proba(X_test)[:, 1]
    results.append({
        'Modelo': name,
        'Acurácia': accuracy_score(y_test, y_pred),
        'Recall (AVC)': recall_score(y_test, y_pred),
        'F1-Score (AVC)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).set_index('Modelo')
results_df = results_df.round(4)
print("Comparação de Modelos:")
print(results_df.to_string())
results_df

In [ ]:
metrics_to_plot = ['Acurácia', 'Recall (AVC)', 'F1-Score (AVC)', 'ROC-AUC']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df.index))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, results_df[metric], width, label=metric)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Modelo')
ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas entre Modelos')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
ACCURACY_MIN = 0.70
RECALL_MIN = 0.10
F1_MIN = 0.10

print("Requisitos mínimos de implantação (alinhados com test_model.py):")
print(f"  Acurácia  >= {ACCURACY_MIN}")
print(f"  Recall    >= {RECALL_MIN}")
print(f"  F1-Score  >= {F1_MIN}")
print()

qualified = results_df[
    (results_df['Acurácia'] >= ACCURACY_MIN) &
    (results_df['Recall (AVC)'] >= RECALL_MIN) &
    (results_df['F1-Score (AVC)'] >= F1_MIN)
]

if not qualified.empty:
    best_model_name = qualified['F1-Score (AVC)'].idxmax()
    print(f"Modelos que atendem aos requisitos: {list(qualified.index)}")
else:
    print("Nenhum modelo atingiu todos os requisitos simultaneamente.")
    fallback = results_df[results_df['Acurácia'] >= ACCURACY_MIN]
    if not fallback.empty:
        best_model_name = fallback['F1-Score (AVC)'].idxmax()
        print(f"Fallback: selecionando melhor F1 entre modelos com acurácia >= {ACCURACY_MIN}")
    else:
        best_model_name = results_df['Acurácia'].idxmax()
        print("Fallback: selecionando modelo com maior acurácia geral")

best_f1 = results_df.loc[best_model_name, 'F1-Score (AVC)']

print(f"\nModelo selecionado: {best_model_name}")
print(f"F1-Score: {best_f1:.4f}")
print(f"\nResumo do melhor modelo:")
print(results_df.loc[best_model_name])

## 13. Análise Final e Conclusões

### Resumo dos Resultados

Neste notebook, construímos e avaliamos quatro modelos de classificação para predição de risco de AVC: **KNN**, **Árvore de Decisão**, **Naive Bayes** e **SVM**. Cada modelo passou por otimização de hiperparâmetros via GridSearchCV com validação cruzada estratificada (5-fold).

### Impacto do Desbalanceamento de Classes

O dataset apresenta um desbalanceamento significativo: apenas **~5% dos registros** correspondem a casos positivos de AVC. Esse desbalanceamento impacta diretamente a performance dos modelos:

- **Acurácia elevada pode ser enganosa:** Um modelo que sempre prevê "sem AVC" atingiria ~95% de acurácia, mas seria completamente inútil para o objetivo clínico.
- **Recall é a métrica mais crítica:** Em contextos médicos, é preferível ter falsos positivos (alertar pacientes saudáveis) do que falsos negativos (não identificar pacientes em risco).
- **F1-Score como compromisso:** Utilizamos o F1-Score como métrica principal de otimização, pois equilibra precisão e recall.

### Análise Comparativa dos Modelos

- **KNN:** Sensível à escala das features e ao número de vizinhos. O desempenho depende fortemente da escolha de K e da métrica de distância.
- **Árvore de Decisão:** Modelo interpretável que pode capturar relações não-lineares, mas propenso a overfitting sem controle de profundidade.
- **Naive Bayes:** Rápido e simples, funciona bem quando a suposição de independência é razoável. Pode ter dificuldades com features correlacionadas.
- **SVM:** Robusto em espaços de alta dimensionalidade, mas pode ser lento para treinar em datasets maiores. O kernel RBF permite capturar fronteiras de decisão complexas.

### Limitações

1. **Tamanho do dataset:** 5.110 registros é relativamente pequeno para treinar modelos robustos de ML.
2. **Desbalanceamento de classes:** A proporção de ~5% de casos positivos limita a capacidade dos modelos de aprender padrões da classe minoritária.
3. **Features limitadas:** O dataset contém apenas variáveis demográficas e clínicas básicas. Dados adicionais (exames laboratoriais, histórico familiar detalhado, estilo de vida) poderiam melhorar a predição.
4. **Valores ausentes:** A coluna `bmi` possui ~201 valores ausentes que foram imputados pela mediana, o que pode introduzir viés.

### Melhorias Futuras

- **SMOTE (Synthetic Minority Over-sampling Technique):** Gerar amostras sintéticas da classe minoritária para balancear o dataset.
- **Métodos Ensemble:** Random Forest, Gradient Boosting (XGBoost, LightGBM) tendem a ter melhor performance em dados tabulares.
- **Mais features:** Integrar dados clínicos adicionais como colesterol, pressão arterial, ECG, entre outros.
- **Deep Learning:** Redes neurais com camadas de atenção podem capturar interações complexas entre features.
- **Threshold Tuning:** Ajustar o limiar de decisão para priorizar recall sobre precisão em contexto clínico.
- **Análise de Feature Importance:** Identificar quais variáveis mais contribuem para a predição.

### Conclusão

Este projeto demonstra o pipeline completo de um projeto de Machine Learning aplicado à saúde: desde a análise exploratória, passando pelo pré-processamento, treinamento e comparação de modelos. Apesar das limitações do dataset, os resultados fornecem uma base sólida para o desenvolvimento do StrokeGuard como ferramenta de apoio à decisão clínica.

## 14. Exportação do Modelo

Exportamos o melhor pipeline completo (pré-processamento + modelo) usando `joblib`. Este arquivo `.pkl` será utilizado pela API Flask do StrokeGuard para realizar predições em tempo real.

In [ ]:
model_map = {
    'KNN': knn_grid,
    'Árvore de Decisão': dt_grid,
    'Naive Bayes': nb_grid,
    'SVM': svm_grid
}

best_pipeline = model_map[best_model_name].best_estimator_
model_filename = 'modelo_treinado.pkl'

joblib.dump(best_pipeline, model_filename)

file_size = os.path.getsize(model_filename) / 1024
print(f"Modelo exportado: {model_filename}")
print(f"Tamanho do arquivo: {file_size:.1f} KB")
print(f"Modelo: {best_model_name}")
print(f"\nO pipeline exportado inclui:")
print("  - Pré-processamento (imputação, padronização, one-hot encoding)")
print("  - Modelo treinado com os melhores hiperparâmetros")
print("\nPara carregar e usar:")
print("  pipeline = joblib.load('modelo_treinado.pkl')")
print("  resultado = pipeline.predict(novo_paciente)")